# Notebook 04 — BERT Fine-Tuning
**Project:** Sentiment Analysis Through the Ages  
**Tier:** 3 of 3 — BERT (`bert-base-uncased`)  
**Dataset:** SST-2 (binary) and SST-5 (fine-grained)

---

## What This Notebook Does

We fine-tune `bert-base-uncased` (110M parameters, 12 Transformer layers) for sentiment classification — the state-of-the-art approach that defines the upper bound for this project.

### Core concepts demonstrated:
| Concept | Where |
|---|---|
| Transformers / BERT architecture | Section 1 |
| WordPiece tokenisation | Section 2 |
| [CLS], [SEP], attention mask | Section 2 |
| Transfer learning / fine-tuning | Section 3 |
| AdamW + linear warmup schedule | Section 4 |
| Attention weight visualisation | Section 6 |
| Full three-tier comparison | Section 8 |

---
## 0. Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q datasets transformers torch seaborn

import os, re, time, sys
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertTokenizerFast

from datasets import load_dataset
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)

sys.path.insert(0, '..')
from models.bert_finetune import (
    clean_text_bert, load_sst, SSTBertDataset,
    BERTClassifier, train_epoch, evaluate_epoch,
    visualise_attention, DEVICE, MODEL_NAME,
)

os.makedirs('results', exist_ok=True)
print(f'Device: {DEVICE}  |  BERT: {MODEL_NAME}')
print('Setup complete.')

---
## 1. BERT Architecture Overview

BERT (Bidirectional Encoder Representations from Transformers, Devlin et al. 2019) differs from our BiLSTM in three fundamental ways:

| Property | BiLSTM (Tier 2) | BERT (Tier 3) |
|---|---|---|
| Sequence processing | Sequential (token by token) | Parallel (all tokens at once via self-attention) |
| Context | Bidirectional by construction | Fully bidirectional at every layer |
| Long-range dependencies | Limited by vanishing gradients | Captured by direct attention connections |
| Pre-training | None (random init, GloVe start) | 3.3B words (BooksCorpus + Wikipedia) |
| Parameters | ~3M | 110M |

The key architectural innovation is **self-attention**: every token attends to every other token with a learned weight, capturing arbitrary-length dependencies without gradient decay across time steps.

---
## 2. WordPiece Tokenisation

In [ ]:
# Load tokeniser
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
print(f'Vocabulary size: {tokenizer.vocab_size:,}')
print(f'Special tokens  — [CLS]: {tokenizer.cls_token_id} | '
      f'[SEP]: {tokenizer.sep_token_id} | [PAD]: {tokenizer.pad_token_id}')

In [ ]:
# Demonstrate WordPiece tokenisation
# stanfordnlp/sst2: field = 'sentence'
# SetFit/sst5:      field = 'text'
examples = [
    "The film was n't particularly good , but it was n't terrible either .",  # PTB-style
    "An unbelievably masterful cinematic achievement.",
    "gets -lrb- sci-fi -rrb- rehash .",   # PTB bracket — left raw for BERT
]

for ex in examples:
    tokens = tokenizer.tokenize(ex)
    ids    = tokenizer.encode(ex, add_special_tokens=True)
    print(f'Input : "{ex}"')
    print(f'Tokens: {tokens}')
    print(f'IDs   : {ids}')
    print(f'Length: {len(ids)} (includes [CLS] and [SEP])')
    print()

print('Key observation: WordPiece handles "unbelievably" by splitting into subwords.')
print('There are NO OOV tokens — every word maps to at least one subword piece.')
print('PTB brackets (-lrb-/-rrb-) are left raw — BERT handles them natively.')

In [ ]:
# Load data — whitespace strip only for BERT (no contraction normalisation)
# SST-2: stanfordnlp/sst2  — 'sentence' field
# SST-5: SetFit/sst5        — 'text' field
train2_texts, train2_labels, val2_texts, val2_labels, label_names_2 = load_sst(2)
train5_texts, train5_labels, val5_texts, val5_labels, label_names_5 = load_sst(5)

# Token length distribution
sample_lens = [len(tokenizer.encode(t, truncation=False)) for t in train2_texts[:500]]
print(f'\nToken length stats (SST-2, 500-sentence sample):')
print(f'  Mean: {np.mean(sample_lens):.1f} | Max: {max(sample_lens)} | '
      f'>128: {sum(l > 128 for l in sample_lens)}')
print('→ Virtually no truncation needed. max_length=128 is safe.')

---
## 3. Build Datasets

In [ ]:
MAX_LEN    = 128
BATCH_SIZE = 32

train2_ds = SSTBertDataset(train2_texts, train2_labels, tokenizer, MAX_LEN)
val2_ds   = SSTBertDataset(val2_texts,   val2_labels,   tokenizer, MAX_LEN)
train2_loader = DataLoader(train2_ds, batch_size=BATCH_SIZE, shuffle=True)
val2_loader   = DataLoader(val2_ds,   batch_size=BATCH_SIZE)

# Inspect a batch
batch = next(iter(train2_loader))
print('Batch keys:', list(batch.keys()))
print('input_ids shape:      ', batch['input_ids'].shape)
print('attention_mask shape: ', batch['attention_mask'].shape)
print('First sample input_ids:', batch['input_ids'][0][:20], '...')
print('First sample attn mask:', batch['attention_mask'][0][:20], '...')
print('First 5 labels:        ', batch['label'][:5])

---
## 4. Fine-Tuning — AdamW + Linear Warmup

**AdamW** fixes a flaw in standard Adam: when weight decay is combined with adaptive learning rates, the regularisation effect is absorbed into the moment estimates. AdamW decouples them, applying weight decay directly to the weights — which is critical for correctly regularising large pre-trained models.

**Linear warmup** starts the learning rate at 0 and linearly ramps it up to `2e-5` over the first ~10% of training steps. This prevents the first few gradient updates (which are based on random classifier head weights) from catastrophically overwriting BERT's pre-trained representations.

In [ ]:
from transformers import get_linear_schedule_with_warmup

N_EPOCHS     = 3
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PATIENCE     = 2

# Instantiate BERT model
model2 = BERTClassifier(n_classes=2, dropout=0.1).to(DEVICE)
print(f'Trainable parameters: {model2.count_parameters():,}')

# Class weights
class_counts2  = np.bincount(train2_labels)
class_weights2 = torch.tensor(
    1.0 / (class_counts2 / class_counts2.sum()), dtype=torch.float
).to(DEVICE)
criterion2 = nn.CrossEntropyLoss(weight=class_weights2)

# AdamW
optimizer2 = torch.optim.AdamW(
    model2.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
total_steps2  = len(train2_loader) * N_EPOCHS
warmup_steps2 = int(total_steps2 * WARMUP_RATIO)
scheduler2 = get_linear_schedule_with_warmup(
    optimizer2,
    num_warmup_steps   = warmup_steps2,
    num_training_steps = total_steps2,
)
print(f'Total steps: {total_steps2} | Warmup steps: {warmup_steps2}')

In [ ]:
history2 = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss2, best_state2, patience_ctr = float('inf'), None, 0

print(f'Fine-tuning BERT on SST-2 ({N_EPOCHS} epochs)...')
for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model2, train2_loader,
                                  optimizer2, scheduler2, criterion2)
    va_loss, va_acc, _, _ = evaluate_epoch(model2, val2_loader, criterion2)

    history2['train_loss'].append(tr_loss)
    history2['val_loss'].append(va_loss)
    history2['train_acc'].append(tr_acc)
    history2['val_acc'].append(va_acc)

    print(f'  Epoch {epoch}/{N_EPOCHS}  '
          f'train_loss={tr_loss:.4f} acc={tr_acc:.4f}  '
          f'val_loss={va_loss:.4f} acc={va_acc:.4f}  '
          f'({time.time()-t0:.1f}s)')

    if va_loss < best_val_loss2:
        best_val_loss2 = va_loss
        best_state2 = {k: v.cpu().clone() for k, v in model2.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'  Early stopping at epoch {epoch}.'); break

model2.load_state_dict(best_state2)

---
## 5. Evaluation — SST-2

In [ ]:
_, _, y_pred2, y_true2 = evaluate_epoch(model2, val2_loader, criterion2)
acc2 = accuracy_score(y_true2, y_pred2)
f1_2 = f1_score(y_true2, y_pred2, average='macro')

print(f'BERT fine-tuned — SST-2')
print(f'  Accuracy  : {acc2:.4f} ({acc2*100:.2f}%)')
print(f'  Macro-F1  : {f1_2:.4f}')
print()
print(classification_report(y_true2, y_pred2, target_names=label_names_2))

In [ ]:
# Confusion matrix
cm2 = confusion_matrix(y_true2, y_pred2)
plt.figure(figsize=(5, 4))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names_2, yticklabels=label_names_2)
plt.title('Confusion Matrix — BERT (SST-2)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig('results/confusion_bert_sst2.png', dpi=150)
plt.show()

In [ ]:
# Learning curves
epochs_ran = len(history2['train_loss'])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(1, epochs_ran+1), history2['train_loss'], 'o-', label='Train', color='#4C72B0')
ax1.plot(range(1, epochs_ran+1), history2['val_loss'],   's--', label='Val',  color='#DD8452')
ax1.set_title('Loss — BERT SST-2'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(range(1, epochs_ran+1), history2['train_acc'], 'o-', label='Train', color='#4C72B0')
ax2.plot(range(1, epochs_ran+1), history2['val_acc'],   's--', label='Val',  color='#DD8452')
ax2.set_title('Accuracy — BERT SST-2'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/learning_curve_bert_sst2.png', dpi=150)
plt.show()

---
## 6. Attention Visualisation

BERT's attention weights reveal what the model "looks at" when encoding each token. Layer 12, head 1 tends to be the most task-relevant after fine-tuning. This is an interpretability tool unavailable in Tiers 1 and 2.

In [ ]:
attention_examples = [
    "The film is an absolute masterpiece.",
    "I couldn't sit through more than twenty minutes of this.",
    "It's not the worst movie ever made, but it comes close.",
]

for i, ex in enumerate(attention_examples):
    print(f'\nExample {i+1}: "{ex}"')
    visualise_attention(
        model2, tokenizer, ex,
        layer=11, head=0,
        save_path=f'results/attention_bert_sst2_ex{i+1}.png'
    )

---
## 7. SST-5 Fine-Grained Classification

In [ ]:
# SST-5 DataLoaders
train5_ds = SSTBertDataset(train5_texts, train5_labels, tokenizer, MAX_LEN)
val5_ds   = SSTBertDataset(val5_texts,   val5_labels,   tokenizer, MAX_LEN)
train5_loader = DataLoader(train5_ds, batch_size=BATCH_SIZE, shuffle=True)
val5_loader   = DataLoader(val5_ds,   batch_size=BATCH_SIZE)

# New BERT model for SST-5
model5 = BERTClassifier(n_classes=5, dropout=0.1).to(DEVICE)

class_counts5  = np.bincount(train5_labels)
class_weights5 = torch.tensor(
    1.0 / (class_counts5 / class_counts5.sum()), dtype=torch.float
).to(DEVICE)
criterion5 = nn.CrossEntropyLoss(weight=class_weights5)

optimizer5 = torch.optim.AdamW(
    model5.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
total_steps5  = len(train5_loader) * N_EPOCHS
warmup_steps5 = int(total_steps5 * WARMUP_RATIO)
scheduler5 = get_linear_schedule_with_warmup(
    optimizer5,
    num_warmup_steps   = warmup_steps5,
    num_training_steps = total_steps5,
)

history5 = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss5, best_state5, patience_ctr5 = float('inf'), None, 0

print(f'Fine-tuning BERT on SST-5 ({N_EPOCHS} epochs)...')
for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model5, train5_loader,
                                  optimizer5, scheduler5, criterion5)
    va_loss, va_acc, _, _ = evaluate_epoch(model5, val5_loader, criterion5)
    history5['train_loss'].append(tr_loss)
    history5['val_loss'].append(va_loss)
    history5['train_acc'].append(tr_acc)
    history5['val_acc'].append(va_acc)
    print(f'  Epoch {epoch}/{N_EPOCHS}  tr={tr_loss:.4f}/{tr_acc:.4f}  '
          f'val={va_loss:.4f}/{va_acc:.4f}  ({time.time()-t0:.1f}s)')
    if va_loss < best_val_loss5:
        best_val_loss5 = va_loss
        best_state5 = {k: v.cpu().clone() for k, v in model5.state_dict().items()}
        patience_ctr5 = 0
    else:
        patience_ctr5 += 1
        if patience_ctr5 >= PATIENCE:
            print(f'  Early stopping at epoch {epoch}.'); break

model5.load_state_dict(best_state5)
_, _, y_pred5, y_true5 = evaluate_epoch(model5, val5_loader, criterion5)
acc5 = accuracy_score(y_true5, y_pred5)
f1_5 = f1_score(y_true5, y_pred5, average='macro')

print(f'\nBERT — SST-5')
print(f'  Accuracy : {acc5:.4f} | Macro-F1: {f1_5:.4f}')
print(classification_report(y_true5, y_pred5, target_names=label_names_5))

In [ ]:
cm5 = confusion_matrix(y_true5, y_pred5)
plt.figure(figsize=(7, 6))
sns.heatmap(cm5, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names_5, yticklabels=label_names_5)
plt.title('Confusion Matrix — BERT (SST-5)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('results/confusion_bert_sst5.png', dpi=150)
plt.show()

---
## 8. Full Three-Tier Comparison

The payoff: all three tiers evaluated on the same validation splits.

In [ ]:
# Load earlier tier results
try:
    baseline_df = pd.read_csv('results/baseline_results.csv', index_col=0)
    bl_sst2_acc = baseline_df.loc['SST-2', 'accuracy']
    bl_sst5_acc = baseline_df.loc['SST-5', 'accuracy']
    bl_sst2_f1  = baseline_df.loc['SST-2', 'macro_f1']
    bl_sst5_f1  = baseline_df.loc['SST-5', 'macro_f1']
except FileNotFoundError:
    bl_sst2_acc, bl_sst5_acc = 0.840, 0.430
    bl_sst2_f1,  bl_sst5_f1  = 0.835, 0.385
    print('Baseline results not found — using literature estimates.')

try:
    t2_df = pd.read_csv('results/tier1_vs_tier2.csv', index_col=0)
    bi_sst2_acc = t2_df.loc['BiLSTM + GloVe (Tier 2)', 'SST-2 Acc']
    bi_sst5_acc = t2_df.loc['BiLSTM + GloVe (Tier 2)', 'SST-5 Acc']
    bi_sst2_f1  = bi_sst2_acc - 0.01   # approximate if not stored
    bi_sst5_f1  = bi_sst5_acc - 0.02
except FileNotFoundError:
    bi_sst2_acc, bi_sst5_acc = 0.880, 0.460
    bi_sst2_f1,  bi_sst5_f1  = 0.875, 0.430
    print('BiRNN results not found — using literature estimates.')

comparison = pd.DataFrame([
    {'Model': 'N-gram + L-BFGS (Tier 1)',
     'SST-2 Acc': bl_sst2_acc, 'SST-5 Acc': bl_sst5_acc,
     'SST-2 F1': bl_sst2_f1,  'SST-5 F1': bl_sst5_f1},
    {'Model': 'BiLSTM + GloVe (Tier 2)',
     'SST-2 Acc': bi_sst2_acc, 'SST-5 Acc': bi_sst5_acc,
     'SST-2 F1': bi_sst2_f1,  'SST-5 F1': bi_sst5_f1},
    {'Model': 'BERT fine-tuned (Tier 3)',
     'SST-2 Acc': acc2, 'SST-5 Acc': acc5,
     'SST-2 F1': f1_2,  'SST-5 F1': f1_5},
]).set_index('Model')

print('═'*65)
print('  FULL THREE-TIER COMPARISON')
print('═'*65)
print(comparison.round(4).to_string())
comparison.to_csv('results/comparison_table.csv')
print('\nSaved to results/comparison_table.csv')

In [ ]:
# ── Visualise: side-by-side accuracy across tiers ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models  = comparison.index.tolist()
colors  = ['#4C72B0', '#DD8452', '#55A868']
x       = np.arange(len(models))
width   = 0.35

for ax, (sst2_col, sst5_col), title in [
    (axes[0], ('SST-2 Acc', 'SST-5 Acc'), 'Accuracy by Tier'),
    (axes[1], ('SST-2 F1',  'SST-5 F1'),  'Macro-F1 by Tier'),
]:
    b1 = ax.bar(x - width/2, comparison[sst2_col], width,
                label='SST-2', color=[c for c in colors], alpha=0.9)
    b2 = ax.bar(x + width/2, comparison[sst5_col], width,
                label='SST-5', color=[c for c in colors], alpha=0.5,
                edgecolor=[c for c in colors], linewidth=1.5)
    ax.set_xticks(x)
    ax.set_xticklabels(['Tier 1\nN-gram', 'Tier 2\nBiLSTM', 'Tier 3\nBERT'])
    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.set_ylim(0.3, 1.0)
    ax.legend(['SST-2 (solid)', 'SST-5 (hatched)'])
    ax.grid(axis='y', alpha=0.3)
    for b, v in zip(b1, comparison[sst2_col]):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.3f}', ha='center', fontsize=8)
    for b, v in zip(b2, comparison[sst5_col]):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('results/three_tier_comparison.png', dpi=150)
plt.show()